# Qwen Image 2.1 — TURBO (8 adım) — MANUEL (ComfyUI arayüzünde elle) — Colab

ComfyUI'yi tünelle açar, grafiği **sen elle sürersin**. Uygulamanın koduna dokunmaz.

- **Madde 482** *(v10 yol haritası)* — ilk bakış: Qwen Image 2.1 nasıl görünüyor, Queen Editor'e yeni bir çizim tarzı olarak girer mi.
- **Madde 484** — sırayla: **Image Edit** şablonunda karakter referansı (portreler resim girişlerine), ana modelin üstünde **NSFW LoRA'ları**, ve **Noct Q V4** checkpoint'i (karakter referansı onunla da).

Her şey baştan iner; sonradan yeni bir LoRA gerekirse ComfyUI yeniden başlatılır.

**Normal model (25 adım) ayrı defterde: `normal.ipynb`.** İkisi ayrı runtime'da koşar, her biri kendi dosyalarını indirir. Turbo, Qwen'in 9 Ekim'de açtığı resmî, 8 adıma damıtılmış checkpoint'i. Noct Q V4'ün Turbo'su yok; burada Comfy-Org'un resmî **Turbo LoRA'sıyla** denenir — bu ikili hiçbir kaynakta denenmedi.

**Input:** prompt ve portreler (ComfyUI UI'da) · **Output:** resim

**Gerekenler:** **A100** runtime. Colab **Secrets**'ta **`CIVITAI_COOKIE`** (zorunlu — LoRA'lar Civitai'den girişle iniyor) ve isteğe bağlı `HF_TOKEN`.

Sıra:
1. **CONFIG** — Civitai çerezi + GPU
2. **Ortak yardımcılar** — log + fail-loud run + model doğrulama
3. **ComfyUI** — `master`'dan; custom node yok, Qwen Image 2.1 ComfyUI'nin kendinde
4. **Modeller** — ~45,2 GB; önce boş disk, sonra Civitai erişimi
5. **Workflow'lar** — resmî `image_qwen_image_2_1_turbo_t2i` ve `image_qwen_image_2_1_turbo_image_edit`; Edit şablonunun örnek resmi
6. **Başlat + cloudflared tünel** → UI linki

> Drive kullanılmaz; modeller her oturumda kaynaktan iner (Colab geçici diski). Üretilen resimler de orada durur: beğendiğini UI'dan indir, runtime kapanınca gider.
>
> **Runtime → Change runtime type → A100.**
>
> **Run all** → en alttaki linke gir → sol kenarda **Workflows** → bir workflow aç → **Run** (Ctrl+Enter).
>
> Hangi dosya neden iniyor: **`indirilecekler.md`** · nasıl kullanılır: **`instructions.md`**

## 1) CONFIG

Doldurulacak bir şey yok → **Run all**.

**`CIVITAI_COOKIE`** Colab **Secrets**'tan okunur — queen-editor'ün okuduğu ad, tek sır bütün defterlere yeter. NSFW LoRA'larının yazarları indirmeyi girişe bağlamış; çerez yoksa ya da kırpılmışsa hücre burada, ilk saniyede durur, 45 GB indirdikten sonra değil. Değeri: civitai.red → giriş → F12 → Application → Cookies → `__Secure-civ-token` → değere **çift tıkla** → Ctrl+A → Ctrl+C (tek tıkla kopyalanan değer kırpık gelir). Ömrü ~30 gün. Değer hiçbir yere basılmaz.

`HF_TOKEN` da Secrets'tan okunur, değeri basılmaz. Yoksa HF indirmeleri token'sız gider: dosyalar açık, ama HF token'sız isteklere 429 dönebiliyor.

GPU burada basılır; GPU yoksa hücre durur.

**Drive mount yok.** Bu deneme Drive kullanmıyor; standardın "önce Drive mount" kuralı Drive kullanan defterler için.

In [ ]:
# === CONFIG ===
COMFY_PORT = 8188
COMFY_ROOT = "/content/ComfyUI"
COMFY_LOG  = "/content/comfyui.log"

import subprocess
from google.colab import userdata

# Required: the NSFW LoRAs' authors gate their downloads behind a Civitai login. Read here, so a missing
# or clipped cookie stops the run in its first second rather than after the Hugging Face downloads.
# The name is queen-editor's: secrets belong to the Google account, so one paste serves every notebook
# (NOTEBOOK-STANDARD, section 4). The value is never printed.
COOKIE_VALUE = (userdata.get("CIVITAI_COOKIE") or "").strip()
# Two asserts, not one: a missing secret and a clipped value are different mistakes.
assert COOKIE_VALUE, "❌ CIVITAI_COOKIE okunamadı — Colab 🔑 Secrets'a ekle ve 'Notebook access'i aç"
assert len(COOKIE_VALUE) > 200, (
    f"❌ CIVITAI_COOKIE çok kısa ({len(COOKIE_VALUE)} karakter) — değer kırpılmış: civitai.red → F12 → "
    "Application → Cookies → __Secure-civ-token, değere çift tıkla → Ctrl+A → Ctrl+C")
print("✓ CIVITAI_COOKIE: Secrets'tan okundu")

# Optional: every HF file is public, but HF can answer 429 to requests without a token. Read once here
# and handed to every download, so huggingface_hub never goes to the vault itself mid-download (in
# queen-editor that late ask timed out and H3 came down unauthenticated into a 429, madde 437).
try:
    HF_TOKEN = (userdata.get("HF_TOKEN") or "").strip()
except Exception as e:
    HF_TOKEN = ""
    print(f"⚠️  HF_TOKEN okunamadı — {type(e).__name__}: {e}\n"
          "    Token'sız inecek: dosyalar açık, ama HF 429 dönebilir.")
if HF_TOKEN:
    print("✓ HF_TOKEN: Secrets'tan okundu")

# === GPU ===
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True)
except FileNotFoundError:
    smi = None
if smi is None or smi.returncode != 0 or not smi.stdout.strip():
    said = (smi.stderr or smi.stdout).strip() if smi else "nvidia-smi bulunamadı"
    raise RuntimeError(f"❌ GPU yok — Runtime → Change runtime type → A100\n{said}")
print(f"\n=== GPU ===\n{smi.stdout.strip()}")

## 2) Ortak Yardımcılar

Tek yerde tanımlanır, sonraki hücreler bunları kullanır:

- `log(msg, level)` — konsola durum satırı
- `run(cmd, label)` — **tek komut-hatası kapısı**: exit ≠ 0 veya timeout → `RuntimeError` + komutun **gerçek stderr** kuyruğu
- `check_safetensors(path)` — **ok** / **partial** / **invalid**. Beklenen boyutu **dosyanın kendi header'ından** hesaplar (tensör `data_offsets`); sunucuya sormaz
- `head_text(path)` — dosyanın ilk byte'ları, **ham**: bozuk bir indirmede orada ne olduğunu gösterir

Kanonik defterden *(`loop_maker/comfy_ui.ipynb`, H3 denemesi üzerinden)* birebir geliyor. H3'ün `strip_unreferenced_tail`'i burada yok: o damga H3'ün nicelemelerini üreten araçtan geliyordu, bu dosyalar Comfy-Org'un kendi deposundan.

In [ ]:
# === Shared helpers — log + fail-loud run + model validation ===
import os, json, time, struct, subprocess

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌", "SKIP": "⏭️ "}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    """Bytes -> human-readable size (e.g. 1.5GB)."""
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def head_text(path, limit=4000):
    """First bytes of a file as raw text — the response body, printed as-is, not interpreted."""
    if not os.path.exists(path):
        return "(dosya yok)"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        text = f.read(limit).decode("utf-8", errors="replace")
    return text + (f"\n… (+{human(size - limit)})" if size > limit else "")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError with the command's own stderr."""
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def check_safetensors(path):
    """State of a model file -> ("ok" | "partial" | "invalid", msg).

    The expected total size is computed from the file itself: a safetensors file is
    [8-byte LE header length][header JSON][tensor data], and the header's data_offsets say where
    the tensor data ends. No Content-Length, no HEAD request (NOTEBOOK-STANDARD, section 3).
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    if size < 8:
        return "invalid", f"too small ({human(size)})"

    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        if not (0 < header_len < 200_000_000):
            return "invalid", f"bad header length ({header_len})"
        if 8 + header_len > size:
            return "partial", f"header incomplete ({human(size)})"
        try:
            header = json.loads(f.read(header_len).decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            return "invalid", f"header parse failed ({type(e).__name__}, {human(size)})"

    ends = [v["data_offsets"][1] for k, v in header.items()
            if k != "__metadata__" and isinstance(v, dict) and "data_offsets" in v]
    if not ends:                        # metadata-only header: nothing to measure against
        return "ok", f"{human(size)}, no tensor offsets"

    expected = 8 + header_len + max(ends)
    if size == expected:
        return "ok", f"{human(size)}, {len(ends)} tensors"
    if size < expected:
        return "partial", f"{size:,} / {expected:,} bytes"
    return "invalid", f"too long: {size:,} / {expected:,} bytes"

print("✓ Ortak yardımcılar hazır (log, run, human, head_text, check_safetensors)")

## 3) ComfyUI — `master`'dan

Qwen Image 2.1'in node'ları (`TextEncodeQwenImage21`, `QwenImage21Cache`) **ComfyUI'nin kendinde**: [PR #16400](https://github.com/Comfy-Org/ComfyUI/pull/16400), 19 Eylül 2026. Custom node da Manager da gerekmiyor; şablonun bütün node'ları çekirdekten.

ComfyUI her koşuda `master`'dan çekilir, sonra klonda `TextEncodeQwenImage21` aranır: yoksa hücre durur — eski bir ComfyUI, grafiği açınca "missing node" olarak değil burada görünür.

`hf_xet` de burada kurulur: HF dosyaları Xet deposunda, ve `hf_hub_download` onunla parçaları doğrudan depodan, paralel çeker. Adresle (`resolve/…`) inen dosya HF'in köprüsünden geçiyor, o da sunucularının çoğunda ~8,7 MB/s'e düşüyor *(queen-editor, madde 438; xet-core #821)*.

Bir komut başarısız olursa hücre `RuntimeError` ile durur ve komutun kendi çıktısını basar.

In [ ]:
# === ComfyUI from master ===
import os

if not os.path.isdir(COMFY_ROOT):
    log("ComfyUI klonlanıyor")
    run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", COMFY_ROOT],
        "clone ComfyUI", timeout=600)
run(["git", "pull", "-q"], "git pull ComfyUI", cwd=COMFY_ROOT)
log("ComfyUI gereksinimleri kuruluyor (birkaç dakika)")
run(["pip", "install", "-q", "-r", "requirements.txt"], "pip install ComfyUI", cwd=COMFY_ROOT)
log("hf_xet kuruluyor")
run(["pip", "install", "-q", "-U", "hf_xet"], "pip install hf_xet")

# Qwen Image 2.1 support is in ComfyUI's own comfy_extras/nodes_qwen.py (PR 16400). Looked for here,
# so a clone without it stops the run now rather than as a missing node in the UI.
commit = run(["git", "log", "-1", "--format=%h %cd"], "git log", cwd=COMFY_ROOT).strip()
QWEN_NODES = f"{COMFY_ROOT}/comfy_extras/nodes_qwen.py"
if not os.path.exists(QWEN_NODES) or "TextEncodeQwenImage21" not in open(QWEN_NODES).read():
    raise RuntimeError(f"❌ Bu ComfyUI'de TextEncodeQwenImage21 yok ({QWEN_NODES}, commit {commit})")
log(f"ComfyUI hazır — {commit}, Qwen Image 2.1 node'ları var", "OK")

## 4) Modeller — ~45,2 GB

**Hugging Face'ten, açık.** [Comfy-Org/Qwen-Image-2.1](https://huggingface.co/Comfy-Org/Qwen-Image-2.1)'dekiler iki resmî şablonun kendi `models` listelerinin istediği dosyalar, ve resmî Turbo LoRA'sı; Noct Q yazarının deposundan, [Noctaluna/Noct-Q-Uncensored-Qwen-Image-2.1](https://huggingface.co/Noctaluna/Noct-Q-Uncensored-Qwen-Image-2.1). Boyutlar HF'in dosya listesinden, bayt bayt.

| Klasör | Dosya | Boyut | |
|---|---|---|---|
| `diffusion_models/` | `qwen_image_2.1_turbo_int8_convrot.safetensors` | 7,26 GB | Turbo model (DiT) |
| `text_encoders/` | `qwen3vl_8b_int8_convrot.safetensors` | 9,35 GB | metin kodlayıcı, normalle ve Noct Q'yla ortak |
| `vae/` | `qwen_image_2.1_vae_bf16.safetensors` | 0,68 GB | VAE, normalle ve Noct Q'yla ortak |
| `text_encoders/` | `qwen3.5_9b_qwen_image_2.1_pe_t2i.int8_convrot.safetensors` | 9,47 GB | t2i şablonunun prompt geliştiricisi — aşağıda neden |
| `text_encoders/` | `qwen3.5_9b_qwen_image_2.1_pe_i2i.int8_convrot.safetensors` | 9,47 GB | Edit şablonunun prompt geliştiricisi — aynı neden |
| `diffusion_models/` | `NoctQ_V4_int8_convrot.safetensors` | 7,26 GB | Noct Q V4, Qwen 2.1'den türetilmiş sansürsüz model (Base) |
| `loras/` | `qwen_image_2.1_turbo_lora_avg_rank_178_bf16.safetensors` | 0,91 GB | resmî Turbo LoRA'sı — Noct Q'yu 8 adımda koşmak için |

**Noct Q neden Turbo LoRA'sıyla.** Noct Q V4 yalnız Base olarak var (25 adım, cfg 3). Comfy-Org'un README'si Turbo için *"use either the full Turbo model, or the Turbo LoRA extracted from the base model. Use 8 steps and cfg 1"* diyor, ve Noct Q Base'in bir merge'ü. Noct Q'nun üstünde Turbo LoRA'sını **hiçbir kaynak denemedi**; deneme gösterir.

**Civitai'den, girişle — `loras/`.** Dört NSFW LoRA'sı, taban modeli "Qwen 2.1" (Base'de eğitilmiş; Turbo'da nasıl çalıştıkları bilinmiyor):

| Dosya | Boyut | Civitai |
|---|---|---|
| `NSFW Qwen by TheseAlpacas V2.safetensors` | 80 MB | [2958918](https://civitai.com/models/2958918), v2.0 |
| `qwen2_1-femaledVaginus-nsfw-v1_3-fine_L22-31.safetensors` | 638 MB | [2986844](https://civitai.com/models/2986844), v1.3b |
| `Qwen_2.1_Unlocked_V1.safetensors` | 50 MB | [2063193](https://civitai.com/models/2063193), Qwen2.1_V1 |
| `nsfw-posture-v3.safetensors` | 84 MB | [2988215](https://civitai.com/models/2988215), v1.0 |

**Prompt geliştiriciler neden iniyor.** İki şablon da onu kapalı açıyor (`refine_prompt: false`), ve kapalıyken hiç yüklenmiyor: arada tembel bir anahtar var. Ama ComfyUI bir işi başlatmadan önce çıktıya bağlı **her** yükleyicinin dosyasını listede arıyor, tembel dal da dahil. Dosya yoksa **Run** `Value not in list` ile duruyor. Şablonlar dokunulmadan çalışsın diye iniyorlar.

Önce **boş disk**: inecek dosyaların toplamı + 5 GiB pay yoksa hiçbir şey inmeden durur. Sonra **Civitai erişimi**: her LoRA'nın ilk 1 KB'ı çerezle istenir; çerez bayatsa hücre Civitai'nin kendi cevabıyla, ağır bir şey inmeden durur.

HF dosyaları `hf_hub_download` ile `/content/hf_stage` altına, Civitai dosyaları `curl` ile (tek bağlantı) `<ad>.part`'a iner, **doğrulanır**, sonra gerçek yerine taşınır: ComfyUI yarım bir dosyayı gerçek adıyla görmez. Doğrulama header'dan: header dosyanın boyunu zaten kanıtlıyor. HF dosyaları ayrıca HF listesindeki bayt sayısına tutulur; Civitai API'sinin sayısı yalnız disk toplamına girer.

Bozuk bir HF dosyası **silinmez**, yerinde kalır ve hücre başını basıp durur. **Yeniden denemeden önce onu elle sil** (hata yolunu basar): silinmezse `hf_hub_download` aynı dosyayı yeniden verir. Civitai'nin `.part`'ı ise yeniden çalıştırınca kendiliğinden ele alınır: yarımsa kaldığı yerden devam eder, tamsa yerine konur, header'ı bozuksa başı basılıp silinir ve baştan iner. Yarıda kalan bir koşuda yerine oturmuş dosyalar atlanır.

In [ ]:
import os, glob, shutil, subprocess, time
from huggingface_hub import hf_hub_download

# === What this notebook downloads ===
# The Comfy-Org files and folders are the ones the official templates' own "models" lists name
# (section 5 checks that against each template it fetches). Sizes are the bytes HF's file list gives:
# the disk check adds them up, and each download is held to its own.
QWEN = "Comfy-Org/Qwen-Image-2.1"
NOCTQ = "Noctaluna/Noct-Q-Uncensored-Qwen-Image-2.1"
MODELS = [
    # (repo, path in the repo, ComfyUI models folder, bytes, label)
    (QWEN, "diffusion_models/qwen_image_2.1_turbo_int8_convrot.safetensors",
     "diffusion_models", 7_256_787_829, "Qwen Image 2.1 Turbo DiT int8"),
    (QWEN, "text_encoders/qwen3vl_8b_int8_convrot.safetensors",
     "text_encoders", 9_350_798_360, "Qwen3-VL 8B metin kodlayıcı int8"),
    (QWEN, "vae/qwen_image_2.1_vae_bf16.safetensors",
     "vae", 675_509_688, "VAE bf16"),
    # The two prompt enhancers, t2i's and Edit's. Off in the templates and never loaded while
    # refine_prompt is off -- the switch in front of them is lazy. But ComfyUI validates every loader
    # upstream of an output before it runs anything, lazy branch or not (execution.py,
    # validate_inputs), so without the file Run stops at "Value not in list". Downloaded so the
    # official templates run as they are.
    (QWEN, "text_encoders/qwen3.5_9b_qwen_image_2.1_pe_t2i.int8_convrot.safetensors",
     "text_encoders", 9_471_072_252, "Prompt geliştirici (t2i) Qwen3.5 9B int8"),
    (QWEN, "text_encoders/qwen3.5_9b_qwen_image_2.1_pe_i2i.int8_convrot.safetensors",
     "text_encoders", 9_471_072_252, "Prompt geliştirici (Edit) Qwen3.5 9B int8"),
    # Noct Q V4 exists only as a merge of the base model; its card names the same text encoder and VAE
    # as above. Comfy-Org's README offers the Turbo LoRA, extracted from the base model, as the other
    # way to Turbo's 8 steps -- so the two come down together here. No source has tried them together.
    (NOCTQ, "NoctQ_V4_int8_convrot.safetensors",
     "diffusion_models", 7_256_784_368, "Noct Q V4 int8"),
    (QWEN, "loras/qwen_image_2.1_turbo_lora_avg_rank_178_bf16.safetensors",
     "loras", 913_314_512, "Turbo LoRA (resmî) bf16"),
]

# Civitai: (model version, file name, bytes, label). Every one's base model is "Qwen 2.1" and every
# one's author asks for a login. The header already proves the length; the API's number (sizeKB) is
# used for the disk sum only.
LORAS = [
    (3357315, "NSFW Qwen by TheseAlpacas V2.safetensors", 79_744_352, "NSFW LORA (TheseAlpacas) v2.0"),
    (3388157, "qwen2_1-femaledVaginus-nsfw-v1_3-fine_L22-31.safetensors", 637_587_392,
     "FemaledVaginus-NSFW v1.3b"),
    (3392016, "Qwen_2.1_Unlocked_V1.safetensors", 50_390_941, "NSFW UNLOCKED Qwen2.1_V1"),
    (3388118, "nsfw-posture-v3.safetensors", 83_943_880, "nsfw-posture-v3"),
]

# hf_hub_download writes the repo's folders and its own .cache under local_dir; neither belongs among
# ComfyUI's models, so a file lands here and is moved into place -- a rename on one disk, not a copy.
STAGE = "/content/hf_stage"
# GiB the disk check keeps free past the models themselves.
HEADROOM_GIB = 5

def target_of(path, folder):
    return f"{COMFY_ROOT}/models/{folder}/{os.path.basename(path)}"

def settled(path, size, label):
    """The verdict's message, or a RuntimeError showing how the file starts. `size` is the count the
    file is held to, or None when the header alone judges. A bad file is never passed on and never
    deleted: it stays where it is, for inspection -- and since hf_hub_download hands a file already in
    STAGE straight back, and a file in place is never fetched again, a re-run needs it deleted by hand
    first."""
    state, msg = check_safetensors(path)
    if state == "ok" and size is not None and os.path.getsize(path) != size:
        state, msg = "invalid", f"{os.path.getsize(path):,} bayt, HF listesi {size:,} diyor"
    if state != "ok":
        raise RuntimeError(f"❌ {label}: {state} — {msg}\n{path}\n--- file head ---\n{head_text(path)}\n"
                           f"Yeniden denemeden önce bu dosyayı elle sil: {path}")
    return msg

def hf_fetch(repo, path, folder, size, label):
    target = target_of(path, folder)
    if os.path.exists(target):
        log(f"{label}: zaten var ({settled(target, size, label)})")
        return
    log(f"{label}: iniyor ({human(size)})")
    start = time.perf_counter()
    try:
        # token=False, not None: None sends huggingface_hub to the vault on its own.
        got = hf_hub_download(repo, path, local_dir=STAGE, token=HF_TOKEN or False)
    except Exception as e:
        said = f"{type(e).__name__}: {e}"
        response = getattr(e, "response", None)
        if response is not None:
            try:
                body = response.text
            except Exception as read_error:   # a streamed body nobody read can raise when asked for
                body = f"(gövde okunamadı — {type(read_error).__name__}: {read_error})"
            said += f"\n--- response: HTTP {response.status_code} ---\n{body}"
        raise RuntimeError(f"❌ {label}: HF {repo}/{path}\n{said}") from None
    msg = settled(got, size, label)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    os.replace(got, target)
    took = time.perf_counter() - start
    log(f"{label}: indirildi ve doğrulandı — {took:.0f} sn, {size / took / 2**20:.1f} MB/s ({msg})", "OK")

# === Civitai (NOTEBOOK-STANDARD, section 4) ===
# civitai.red, where the cookie was issued: sent to .com it is cross-site and the answer is the login
# page. The login cookie only -- a ?token= API key authenticates as that key's account and gets 401 on
# a gated file. The cookie rides in curl's arguments, so an error prints curl's stderr, never the
# command line.
def civitai_url(version_id):
    return f"https://civitai.red/api/download/models/{version_id}"

def cookie_header():
    return f"Cookie: __Secure-civ-token={COOKIE_VALUE}"

PROBE = "/content/_probe.bin"

def civitai_probe(version_id, label):
    """The first KB of a Civitai file, asked with the cookie before anything heavy comes down: a stale
    or clipped cookie stops the run here, with Civitai's own answer. The flags are queen-editor's:
    exit 28 is curl's timeout, which --limit-rate and --max-time can bring on a good probe too, so the
    code and the bytes decide. A safetensors file's header JSON starts at its ninth byte; a login page
    or an error JSON does not."""
    done = subprocess.run(["curl", "-sS", "-L", "--max-time", "20", "--limit-rate", "200k",
                           "-r", "0-1023", "-H", cookie_header(), "-w", "%{http_code}", "-o", PROBE,
                           civitai_url(version_id)], capture_output=True, text=True)
    if done.returncode not in (0, 28):
        raise RuntimeError(f"❌ {label}: Civitai probe — curl exit {done.returncode}\n{done.stderr.strip()}")
    code = (done.stdout or "").strip()[-3:]
    raw, said = b"", ""
    if os.path.exists(PROBE):
        with open(PROBE, "rb") as f:
            raw = f.read(10)
        said = head_text(PROBE)
        os.remove(PROBE)
    if code.startswith("2") and raw[8:10] == b'{"':
        log(f"{label}: Civitai erişimi OK", "OK")
        return
    raise RuntimeError(f"❌ {label}: Civitai probe — HTTP {code}\n"
                       f"--- Civitai yanıtı ---\n{said.strip() or '(boş gövde)'}"
                       + (f"\n--- curl ---\n{done.stderr.strip()}" if done.stderr.strip() else ""))

def civitai_fetch(version_id, filename, label):
    target = target_of(filename, "loras")
    if os.path.exists(target):
        log(f"{label}: zaten var ({settled(target, None, label)})")
        return
    part = target + ".part"
    os.makedirs(os.path.dirname(target), exist_ok=True)
    # A .part an earlier run left is judged before curl touches it, as queen-editor's fetch does.
    state = None
    if os.path.exists(part):
        state, msg = check_safetensors(part)
        if state == "invalid":
            # NOTEBOOK-STANDARD section 3 keeps a bad file for inspection, but these bytes do not begin
            # with a safetensors header, so nothing appended to them can make a model of them, and
            # their head is printed right here. They go, and the file comes down fresh.
            log(f"{label}: .part bozuk ({msg}) — siliniyor, baştan iniyor\n"
                f"--- dosyanın başı ---\n{head_text(part)}", "WARN")
            os.remove(part)
        elif state == "ok":
            log(f"{label}: .part zaten tam ({msg}) — indirilmiyor")
        else:
            log(f"{label}: .part'tan devam ({msg})")
    start = time.perf_counter()
    before = os.path.getsize(part) if os.path.exists(part) else 0
    if state != "ok":
        log(f"{label}: Civitai'den iniyor")
        # One connection, by curl: Civitai redirects to its store, the B2 store answers 403 when the
        # cookie comes along, and curl drops it at the change of host where aria2c carries it. -C -
        # carries on from a partial .part.
        try:
            run(["curl", "-sS", "-L", "-C", "-", "--fail-with-body", "--max-time", "3600",
                 "-H", cookie_header(), "-o", part, civitai_url(version_id)], label, timeout=3700)
        except RuntimeError as e:
            # The .part stays: what came down before the failure may be good, and the next run judges
            # it and resumes.
            raise RuntimeError(f"❌ {e}\n{civitai_url(version_id)}\n"
                               f"--- .part'ın başı ---\n{head_text(part)}") from None
    state, msg = check_safetensors(part)
    if state != "ok":
        raise RuntimeError(f"❌ {label}: {state} — {msg}\n{part}\n--- dosyanın başı ---\n{head_text(part)}")
    os.replace(part, target)
    took, size = time.perf_counter() - start, os.path.getsize(target) - before
    log(f"{label}: indirildi ve doğrulandı — {took:.0f} sn, {size / took / 2**20:.1f} MB/s ({msg})", "OK")

# === Free disk, before anything downloads ===
need = (sum(size for _, path, folder, size, _ in MODELS if not os.path.exists(target_of(path, folder)))
        + sum(size for _, filename, size, _ in LORAS if not os.path.exists(target_of(filename, "loras"))))
free = shutil.disk_usage("/content").free
log(f"İnecek: {need / 1024**3:.1f} GiB | Diskte boş: {free / 1024**3:.1f} GiB")
if free < need + HEADROOM_GIB * 1024**3:
    raise RuntimeError(f"❌ Disk yetmiyor: {need / 1024**3:.1f} GiB model + {HEADROOM_GIB} GiB pay "
                       f"gerekiyor, {free / 1024**3:.1f} GiB boş.")

# === Civitai access, before anything heavy ===
for version_id, filename, _, label in LORAS:
    if not os.path.exists(target_of(filename, "loras")):
        civitai_probe(version_id, label)

# === Downloads ===
for repo, path, folder, size, label in MODELS:
    hf_fetch(repo, path, folder, size, label)
for version_id, filename, _, label in LORAS:
    civitai_fetch(version_id, filename, label)

# === Summary (reaching here means every model downloaded + validated) ===
print("")
for folder in sorted({folder for _, _, folder, _, _ in MODELS} | {"loras"}):
    print(f"📂 {folder}/")
    for f in sorted(glob.glob(f"{COMFY_ROOT}/models/{folder}/*.safetensors")):
        print(f"   {human(os.path.getsize(f)):>8s}  {os.path.basename(f)}")
log("Tüm modeller indirildi ve doğrulandı", "OK")

## 5) Workflow'lar — Workflows paneline

Hiçbir grafik elle yazılmadı ve üretilmedi: Comfy-Org'un resmî şablonları kaynağından **olduğu gibi** iner ve ComfyUI'nin kullanıcı workflow klasörüne (`/content/ComfyUI/user/default/workflows/`) konur. UI'da sol kenardaki **Workflows** panelinde tek tıkla açılır.

| Workflow | Ne için | Kaynak |
|---|---|---|
| `image_qwen_image_2_1_turbo_t2i` | prompt'tan resim | [Comfy-Org, resmî](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/image_qwen_image_2_1_turbo_t2i.json) |
| `image_qwen_image_2_1_turbo_image_edit` | karakter referansı: portreler resim girişlerine | [Comfy-Org, resmî](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/image_qwen_image_2_1_turbo_image_edit.json) |

Noct Q'nun kendi workflow'u bu defterde yok: Base ayarlarıyla geliyor (25 adım, cfg 3); normal defterde.

Bayt bayt aynı yazılır; okunur ama değiştirilmez. Konmadan önce iki kontrol: dosya JSON mu, ve şablonun `models` listesinde adı geçen **her** dosya 4. bölümde inmiş mi. Şablon bir gün başka bir dosya isterse burada durur ve hangisi olduğunu söyler — UI'da "missing model" olarak değil.

**Örnek resim.** Edit şablonunun `Load Image` node'u Comfy'nin bir örnek resmini adıyla istiyor, ve şablonun kendi *"Input Asset"* notu adresini veriyor: [`ballet_dancer_photography_minimalist_architecture.png`](https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/input/ballet_dancer_photography_minimalist_architecture.png). Resim `ComfyUI/input/`'ta yoksa **Run** onu yükleyemez ve durur; o yüzden iner — şablon önce olduğu gibi koşar, sonra portrelerini koyarsın.

Hedefte dosya zaten varsa hiçbir şey indirilmez ve dosyaya dokunulmaz: UI'da aynı adla kaydettiğin bir grafik ya da yüklediğin bir resim, hücre tekrar çalışınca ezilmez.

In [ ]:
# === Workflows -> ComfyUI's user workflows, and the Edit template's input image ===
import os, json

COMFY_TEMPLATES = "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main"
# (file name, raw address): Comfy-Org's official templates, each saved as it came, none written or
# changed here.
TEMPLATES = [
    ("image_qwen_image_2_1_turbo_t2i.json",
     f"{COMFY_TEMPLATES}/templates/image_qwen_image_2_1_turbo_t2i.json"),
    ("image_qwen_image_2_1_turbo_image_edit.json",
     f"{COMFY_TEMPLATES}/templates/image_qwen_image_2_1_turbo_image_edit.json"),
]
# The Edit template's Load Image node names this, and its "Input Asset" note says where it is. Without
# it in ComfyUI/input, Run stops at the Load Image, so it comes down for the template to run as it is
# before the user's own portraits go in.
INPUT_IMAGES = ["ballet_dancer_photography_minimalist_architecture.png"]
# Where ComfyUI keeps the user's saved workflows; the UI's Workflows panel lists this folder.
WORKFLOWS = f"{COMFY_ROOT}/user/default/workflows"
INPUT = f"{COMFY_ROOT}/input"
PNG = b"\x89PNG\r\n\x1a\n"

def named_models(node):
    """Every {"name", "directory"} entry in the workflow -- the loaders' own "models" lists."""
    if isinstance(node, dict):
        if isinstance(node.get("name"), str) and isinstance(node.get("directory"), str):
            yield node["directory"], node["name"]
        for value in node.values():
            yield from named_models(value)
    elif isinstance(node, list):
        for value in node:
            yield from named_models(value)

def curl_part(url, saved, label):
    """The file at `url` into saved + ".part" -- the caller checks it and renames it into place."""
    part = saved + ".part"
    try:
        run(["curl", "-sS", "-L", "--fail-with-body", "--max-time", "60", "-o", part, url],
            label, timeout=120)
    except RuntimeError as e:
        raise RuntimeError(f"❌ {e}\n{url}\n--- dosyanın başı ---\n{head_text(part)}") from None
    return part

def place_workflow(name, url):
    saved = f"{WORKFLOWS}/{name}"
    # A file already there may be the user's own save under the same name: a re-run leaves it alone.
    if os.path.exists(saved):
        log(f"Workflow zaten var, olduğu gibi bırakıldı: Workflows → {name[:-5]} ({saved})")
        return
    part = curl_part(url, saved, f"workflow {name}")
    try:
        with open(part, "rb") as f:
            graph = json.loads(f.read())
    except ValueError as e:
        raise RuntimeError(f"❌ {name}: JSON değil — {e}\n{url}\n--- dosyanın başı ---\n{head_text(part)}") from None
    wanted = sorted(set(named_models(graph)))
    missing = [f"models/{folder}/{file}" for folder, file in wanted
               if not os.path.exists(f"{COMFY_ROOT}/models/{folder}/{file}")]
    if not wanted or missing:
        raise RuntimeError(f"❌ Workflow'un istediği dosyalar inenlerle tutmuyor ({url})\n"
                           f"İstediği: {wanted or '(model listesi bulunamadı)'}\n"
                           f"Diskte olmayan: {missing}")
    # The bytes as they came, renamed into place only after both checks.
    os.replace(part, saved)
    log(f"Workflow hazır: Workflows → {name[:-5]} ({len(wanted)} model dosyası, hepsi diskte)", "OK")

def place_image(name):
    saved = f"{INPUT}/{name}"
    # A picture already there may be one the user uploaded under the same name.
    if os.path.exists(saved):
        log(f"Örnek resim zaten var: {saved}")
        return
    url = f"{COMFY_TEMPLATES}/input/{name}"
    part = curl_part(url, saved, f"resim {name}")
    with open(part, "rb") as f:
        head = f.read(len(PNG))
    if head != PNG:
        raise RuntimeError(f"❌ {name}: PNG değil\n{url}\n--- dosyanın başı ---\n{head_text(part)}")
    os.replace(part, saved)
    log(f"Örnek resim hazır: {saved} ({human(os.path.getsize(saved))})", "OK")

os.makedirs(WORKFLOWS, exist_ok=True)
os.makedirs(INPUT, exist_ok=True)
for name, url in TEMPLATES:
    place_workflow(name, url)
for name in INPUT_IMAGES:
    place_image(name)

## 6) Başlat + cloudflared Tünel

ComfyUI arka planda başlar (90 sn içinde `/system_stats` cevap vermezse log'un son 30 satırını basıp durur), tünel linki basılır, sonra hücre **bilerek açık kalır** — biterse Colab runtime'ı boşta sayıp tüneli öldürür. ComfyUI'nin canlı log'u bu hücrede akar.

In [ ]:
import subprocess, time, urllib.request, re, os

if not os.path.isfile("/content/cloudflared"):
    run(["wget", "-q", "-O", "/content/cloudflared",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], "cloudflared")
    run(["chmod", "+x", "/content/cloudflared"], "chmod cloudflared")

# Re-run safety: kill the previous ComfyUI + tunnel before starting new ones
subprocess.run(["pkill", "-f", "main.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

logf = open(COMFY_LOG, "w")
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
                 cwd=COMFY_ROOT, stdout=logf, stderr=subprocess.STDOUT)
ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open(COMFY_LOG).readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")
log(f"ComfyUI ayakta ({(i + 1) * 2}s)", "OK")

# cloudflared output goes to a file, not a pipe: an unread pipe fills up and blocks the process.
# http2, not cloudflared's default QUIC: QUIC rides on UDP, which Colab's network throttles. Queen
# Editor measured it on 2026-08-24: the same photo took 17.74 s over QUIC and 0.18 s over http2.
tunlog = "/content/cloudflared.log"
subprocess.Popen(["/content/cloudflared", "tunnel", "--protocol", "http2", "--url", f"http://127.0.0.1:{COMFY_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read()) if os.path.exists(tunlog) else None
    if m:
        link = m.group(0)
        break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"\n🔗 ComfyUI linki: {link}\n")
print("⬆️  Linke gir → sol kenarda Workflows:")
print("    image_qwen_image_2_1_turbo_t2i         — prompt'tan resim")
print("    image_qwen_image_2_1_turbo_image_edit  — karakter referansı (portreler image_1, image_2, …)")
print("    LoRA'lar models/loras/'ta; nasıl eklenir, Noct Q'ya nasıl geçilir: instructions.md")
print("\n⚠️  cfg 1'de negatif prompt hiçbir şey yapmaz — şablonun kendi notu\n")

# === Keep the cell OPEN (critical) ===
# ComfyUI + the tunnel run in the background. If this cell ENDS, Colab can call the runtime idle
# and disconnect -> ComfyUI + link die. Streaming the log keeps the cell in the foreground and
# shows generation progress when Run is pressed in the UI.
# To stop: interrupt this cell (■) or Runtime -> Disconnect.
print("📡 ComfyUI çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", COMFY_LOG])
except KeyboardInterrupt:
    log("Hücre durduruldu — ComfyUI hâlâ arka planda çalışıyor (yeni link için bu hücreyi tekrar çalıştır).", "WARN")